# CT4110 – Topic 7 Assignment
## Simon's Algorithm

This exercise has three tasks. In **Task 1** you construct a two-qubit Simon oracle and recover its hidden string. In **Task 2** the three-qubit oracle is provided; you complete Simon's algorithm, measure the input register, and recover the hidden string. **Task 3** investigates classical period finding and measures how its work grows with the input size.

Complete every `YOUR CODE HERE` and `ANSWER: [ ]` section. Run the notebook from top to bottom before submission.

In [ ]:
# Course setup – run this cell first
try:
    import qiskit
    import qiskit_aer
except ImportError:
    %pip install -q qiskit qiskit-aer matplotlib pylatexenc

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
print("Qiskit setup complete.")

## Simon's algorithm: detailed roadmap

Simon's problem starts with a promised two-to-one function $f:\{0,1\}^n\rightarrow\{0,1\}^n$. A nonzero hidden string $s$ pairs every input $x$ with $x\oplus s$:

$$f(x)=f(x\oplus s).$$

A Simon circuit uses two $n$-qubit registers. The input register holds $x$, and the output register receives $f(x)$. The algorithm proceeds as follows:

1. Start in $|0\rangle^{\otimes n}|0\rangle^{\otimes n}$.
2. Apply Hadamard gates to the input register, creating an equal superposition of all $2^n$ inputs.
3. Apply the oracle $U_f$. The two registers become entangled because each $x$ is paired with its value $f(x)$.
4. Apply a second Hadamard layer to the input register. Amplitudes cancel whenever the candidate output $z$ has an odd binary inner product with $s$.
5. Measure only the input register. Every observed string $z$ therefore obeys

$$z\cdot s=0\pmod 2.$$

One measurement supplies one linear equation. Repeat the circuit until you have $n-1$ independent equations, solve them for $s$, and exclude the all-zero string using Simon's promise. The output register does not need to be measured.

The lecture reads a register from the top wire downward, using big-endian order. Qiskit uses the opposite wire convention. In both Simon tasks, first measure each input qubit into the classical bit with the same index. Then call `reverse_bits()` on the measured circuit. Qiskit reverses the quantum and classical wires together, so the measurement connections stay intact without manual classical-bit rewiring.

## Task 1 – Construct a two-qubit Simon oracle

You must construct the oracle. It maps the two-bit input $x$ to the following two-bit function value:

| $x$ | $f(x)$ |
|---|---|
| 00 | 00 |
| 01 | 01 |
| 10 | 01 |
| 11 | 00 |

Use four qubits. Qubits `0` and `1` form the input register. Qubits `2` and `3` form the output register. Build a reversible oracle that performs $|x\rangle|y\rangle \mapsto |x\rangle|y\oplus f(x)\rangle$.

**Hint:** the first output bit, shown on the left, always remains `0`. The second output bit, shown on the right, stores the XOR of the two input bits.

In [ ]:
# Task 1 checkpoint 1 – construct the oracle
def task1_oracle():
    oracle = QuantumCircuit(4, name="U_f task 1")
    # Add the gates that compute f(x) into the output register.
    # YOUR CODE HERE
    return oracle

task1_oracle().draw("mpl")

# REQUIRED OUTPUT CHECK
# The oracle diagram must use both input qubits as controls, with q3 as
# the target because q3 is the second output bit in lecture wire order.

### Task 1 circuit and measurement

Construct Simon's circuit in this order:

1. Apply Hadamard gates to both input qubits.
2. Append your oracle to all four qubits.
3. Apply Hadamard gates to both input qubits again.
4. Measure input qubits `[0, 1]` into classical bits `[0, 1]`.
5. Call `reverse_bits()` on the measured circuit and store the returned circuit as `task1_big_endian`. The quantum and classical wires reverse together, so the displayed result follows the lecture's top-wire-first order.
6. Run 1,000 shots and plot the counts.

In [ ]:
# Task 1 checkpoint 2 – implement and run Simon's algorithm
task1_qc = QuantumCircuit(4, 2)

# YOUR CODE HERE

# Measure input qubits [0, 1] into classical bits [0, 1].
# Then call reverse_bits() on the measured circuit and store the returned
# circuit as task1_big_endian. Do not manually rewire the classical bits.
# YOUR CODE HERE

task1_big_endian.draw("mpl")

# Transpile and run task1_big_endian for 1,000 shots.
# Store the counts in task1_counts.
# YOUR CODE HERE

print(task1_counts)
plot_histogram(task1_counts)

# REQUIRED OUTPUT CHECK
# A counts dictionary must be printed and a measurement histogram must appear.
# Do not interpret the measured bit strings until the decoding step below.

In [ ]:
# TASK 1 DECODING
# You may complete this decoding with pen and paper.
# We do not expect you to solve the equations programmatically.
# Type your answers and working below as Python comments beginning with #.
#
# Ignore the uninformative outcome 00. For each nonzero measured string
# After reverse_bits(), read the measured outcome from left to right as
# Z0 Z1. Write the equation Z0*S0 + Z1*S1 = 0 (mod 2).
# Use that equation and Simon's promise S != 00 to recover S.
#
# Measured nonzero outcome(s):
# ANSWER: 
#
# Equation modulo 2:
# ANSWER: 
#
# Hidden string S = S0 S1:
# ANSWER: 

## Task 2 – Complete a three-qubit Simon circuit

The oracle below is provided. Before bit reversal, it uses input qubits `0, 1, 2` and output qubits `3, 4, 5`. Do not change the oracle. Complete the Simon circuit around it.

In [ ]:
# Provided three-qubit Simon oracle
def task2_oracle():
    oracle = QuantumCircuit(6, name="U_f task 2")
    oracle.cx(0, 3)
    oracle.cx(2, 3)
    oracle.cx(1, 4)
    return oracle

task2_oracle().draw("mpl")

In [ ]:
# Task 2 checkpoint – complete and run Simon's algorithm
task2_qc = QuantumCircuit(6, 3)

# Apply the first Hadamard layer to the input register.
# Append task2_oracle() to all six qubits.
# Apply the second Hadamard layer to the input register.
# Measure input qubits [0, 1, 2] into classical bits [0, 1, 2].
# Then call reverse_bits() on the measured circuit and store the result
# as task2_big_endian. Do not manually rewire the classical bits.
# YOUR CODE HERE

task2_big_endian.draw("mpl")

# Run task2_big_endian for 2,000 shots, store the result in task2_counts, print it,
# and display a histogram.
# YOUR CODE HERE

print(task2_counts)
plot_histogram(task2_counts)

# REQUIRED OUTPUT CHECK
# Exactly four three-bit outcomes should have nonzero counts.

In [ ]:
# TASK 2 DECODING
# You may complete this decoding with pen and paper.
# We do not expect you to solve the equations programmatically.
# Type your answers and working below as Python comments.
#
# Choose two linearly independent nonzero outcomes from task2_counts.
# After reverse_bits(), read each measured outcome from left to right as
# Z0 Z1 Z2, and write the hidden string as S0 S1 S2.
# Each outcome gives Z0*S0 + Z1*S1 + Z2*S2 = 0 (mod 2).
# Solve the two equations over bits and exclude S = 000.
#
# Two independent outcomes:
# ANSWER: 
#
# Two equations modulo 2:
# ANSWER: 
#
# Hidden string S = S0 S1 S2:
# ANSWER: 
#
# Verification: list all four outcomes satisfying Z dot S = 0 (mod 2)
# and compare them with the histogram.
# ANSWER: 

## Task 3 – Classical period finding and its scaling

For integers $a$ and $N$ with $\gcd(a,N)=1$, the order $r$ is the smallest positive integer satisfying

$$a^r=1\pmod N.$$

The sequence $f(k)=a^k\bmod N$ repeats with period $r$. A direct classical search repeatedly performs two separate operations:

1. **Multiplication:** multiply the current value by $a$.
2. **Modulo reduction:** divide the product by $N$ and keep only the remainder.

For example, if the current value is 5, $a=5$, and $N=21$, first calculate $5\times5=25$. Then calculate $25\bmod21=4$. The value carried into the next iteration is 4, not 25.

The search stops when the reduced value returns to 1. This requires exactly $r$ multiplication-and-modulo steps. Since an $n$-bit modulus can have $N\approx2^n$ and the order can be as large as $N-1$, this direct method can require $O(2^n)$ steps.

This task studies that classical cost only. You are not required to implement the quantum order-finding circuit.

In [ ]:
# Task 3 checkpoint 1 – direct classical order finding
from math import gcd

def classical_order(a, N):
    # Check that a and N are coprime.
    # Start with value = 1. For each exponent, do these separately:
    #   1. product = value * a
    #   2. reduced_value = product % N
    # Count one multiplication each time product = value * a is evaluated.
    # Return (period, number_of_multiplications).
    # YOUR CODE HERE
    pass

period_5_21, multiplications_5_21 = classical_order(5, 21)
print("period of 5 modulo 21:", period_5_21)
print("number of multiplications:", multiplications_5_21)

# REQUIRED OUTPUT CHECK
# The period and number of multiplications for (5, 21)..

### Scaling experiment

Use your `classical_order` function for the explicit `(a, N)` cases provided below. The cases use different composite moduli covering bit lengths 4 through 16. Their periods do not double exactly at every step, but the overall multiplication count should rise rapidly as the number of bits increases. Your job is to calculate the period and multiplication count for every case, print a scaling table, and plot multiplication count against bit length.

In [ ]:
# Task 3 checkpoint 2 – measure and plot the classical scaling
import matplotlib.pyplot as plt

cases = [
    (2, 9),       # 4-bit modulus
    (2, 25),      # 5-bit modulus
    (3, 49),      # 6-bit modulus
    (2, 121),     # 7-bit modulus
    (2, 243),     # 8-bit modulus
    (2, 361),     # 9-bit modulus
    (3, 961),     # 10-bit modulus
    (3, 1849),    # 11-bit modulus
    (2, 3721),    # 12-bit modulus
    (3, 7921),    # 13-bit modulus
    (3, 16129),   # 14-bit modulus
    (2, 32761),   # 15-bit modulus
    (6, 63001),   # 16-bit modulus
]

bit_lengths = []
multiplication_counts = []
results = []

# For every (a, N), call classical_order and store a row containing:
# a, N, N.bit_length(), period and number of multiplications.
# Print a heading and every row as a readable scaling table.
# YOUR CODE HERE

# Plot multiplication_counts against bit_lengths. Label both axes,
# add a title and show the plot.
# YOUR CODE HERE

# REQUIRED OUTPUT CHECK
# The table must contain values for bit lengths 4 through 16.
# The plot should show a strong upward trend as the bit length increases.